# Phase 3 : nettoyage et transformation

Ce notebook part des fichiers bruts et produit les tables nettoyées. Il applique dans l'ordre les fonctions de `src/clean_data.py`. Chaque étape est expliquée dans `docs/`.

## Chargement des données (P3-01)

In [1]:
import time
import pandas as pd

from src.load_data import (
    load_transactions, load_cards, load_users, load_mcc_codes, load_fraud_labels,
    load_clean_table,
)
from src.clean_data import (
    convert_currency_column, report_conversion_issues, convert_month_year_column,
    check_datetime_column, build_mcc_table, check_mcc_join, add_mcc_families,
    check_mcc_families, transform_errors_column, check_errors_transformation,
    structure_transaction_location, check_transaction_location,
    flag_suspicious_transactions, flag_suspicious_users, flag_suspicious_cards,
    check_suspicious_flags, COLUMNS_TO_DROP, drop_columns, check_columns_removed,
    save_clean_tables, transformation_summary,
)
from src.diagnostic import count_error_types, check_clean_tables

pd.set_option("display.max_colwidth", None)

In [2]:
df_transaction = load_transactions()
df_card = load_cards()
df_users = load_users()
df_mcc = load_mcc_codes()
df_labels = load_fraud_labels()

# État de départ, pour les contrôles (P3-11) et le journal (P3-12)
etat_avant = {name: (len(df), list(df.columns)) for name, df in [
    ("transactions", df_transaction), ("cards", df_card), ("users", df_users),
    ("mcc", df_mcc), ("fraud_labels", df_labels)]}

# Référence des types d'erreur (P2-13), mesurée avant toute transformation
ref_erreurs = count_error_types(df_transaction)

for name, df in [("transactions", df_transaction), ("cards", df_card), ("users", df_users),
                 ("mcc", df_mcc), ("fraud_labels", df_labels)]:
    print(f"{name}: {df.shape[0]:,} lignes, {df.shape[1]} colonnes")
print(f"Mémoire des transactions : {df_transaction.memory_usage(deep=True).sum() / 1e6:.1f} Mo")

transactions: 13,305,915 lignes, 12 colonnes
cards: 6,146 lignes, 13 colonnes
users: 2,000 lignes, 14 colonnes
mcc: 109 lignes, 2 colonnes
fraud_labels: 8,914,963 lignes, 2 colonnes
Mémoire des transactions : 1371.0 Mo


**Constats**

- Les cinq tables sont chargées : 13 305 915 transactions, 6 146 cartes, 2 000 clients, 109 codes MCC et 8 914 963 étiquettes.
- La table des transactions occupe 1 371,0 Mo en mémoire.

## P3-02 : conversion des montants

Critère de fin : aucune valeur non convertie, ou cas listés. Voir `docs/P3-02_conversion_montants.md`.

In [3]:
# transactions_data
original = df_transaction["amount"].copy()
df_transaction["amount"] = convert_currency_column(df_transaction["amount"])
report_conversion_issues(original, df_transaction["amount"], "transactions.amount")

# cards_data
original = df_card["credit_limit"].copy()
df_card["credit_limit"] = convert_currency_column(df_card["credit_limit"])
report_conversion_issues(original, df_card["credit_limit"], "cards.credit_limit")

# users_data
for col in ["per_capita_income", "yearly_income", "total_debt"]:
    original = df_users[col].copy()
    df_users[col] = convert_currency_column(df_users[col])
    report_conversion_issues(original, df_users[col], f"users.{col}")
del original

transactions.amount: 0 valeur(s) non convertie(s)
cards.credit_limit: 0 valeur(s) non convertie(s)
users.per_capita_income: 0 valeur(s) non convertie(s)
users.yearly_income: 0 valeur(s) non convertie(s)
users.total_debt: 0 valeur(s) non convertie(s)


**Constats**

- Les cinq colonnes de montants sont converties : 0 valeur non convertie dans chacune.

## P3-03 : conversion des dates

Critère de fin : aucune date non convertie. Voir `docs/P3-03_conversion_dates.md`.

In [4]:
original = df_card["expires"].copy()
df_card["expires"] = convert_month_year_column(df_card["expires"], end_of_month=True)
report_conversion_issues(original, df_card["expires"], "cards.expires")

original = df_card["acct_open_date"].copy()
df_card["acct_open_date"] = convert_month_year_column(df_card["acct_open_date"])
report_conversion_issues(original, df_card["acct_open_date"], "cards.acct_open_date")
del original

check_datetime_column(df_card["expires"], "cards.expires")
check_datetime_column(df_card["acct_open_date"], "cards.acct_open_date")
check_datetime_column(df_transaction["date"], "transactions.date")

cards.expires: 0 valeur(s) non convertie(s)
cards.acct_open_date: 0 valeur(s) non convertie(s)
cards.expires: type=datetime64[ns], manquantes=0, min=1997-07-31 00:00:00, max=2024-12-31 00:00:00
cards.acct_open_date: type=datetime64[ns], manquantes=0, min=1991-01-01 00:00:00, max=2020-02-01 00:00:00
transactions.date: type=datetime64[ns], manquantes=0, min=2010-01-01 00:01:00, max=2019-10-31 23:59:00


**Constats**

- `expires` et `acct_open_date` sont converties : 0 valeur non convertie.
- Aucune date ne manque dans les trois colonnes de dates.
- Les transactions vont du 1er janvier 2010 au 31 octobre 2019. Les ouvertures de compte vont de janvier 1991 à février 2020. Les expirations vont de juillet 1997 à décembre 2024.

## P3-04 et P3-05 : table des catégories MCC et familles

Critères de fin : jointure avec les transactions sans perte ; table de correspondance validée. Voir `docs/P3-04_table_mcc.md` et `docs/P3-05_familles_mcc.md`.

In [5]:
df_mcc = build_mcc_table(df_mcc)
ok_jointure = check_mcc_join(df_transaction, df_mcc)
print("Jointure sans perte :", ok_jointure)

df_mcc = add_mcc_families(df_mcc)
ok_familles = check_mcc_families(df_mcc, df_transaction)
print("Table de correspondance valide :", ok_familles)

Lignes avant : 13,305,915 | après : 13,305,915
Transactions sans catégorie : 0
Jointure sans perte : True
Codes sans famille : []
Codes sans description française : []
Codes du dictionnaire absents de la table : []
                                          n_codes  n_transactions  part_%
famille                                                                  
Transport, carburant et automobile             15         3352312    25.2
Alimentation                                    3         3182334    23.9
Restauration et bars                            3         1747967    13.1
Grandes surfaces et commerces divers            5         1332087    10.0
Santé et soins                                  8          886034     6.7
Services professionnels et financiers           7          718301     5.4
Loisirs, culture et divertissement             13          638844     4.8
Services publics et télécommunications          5          626682     4.7
Maison, jardin et électronique               

**Constats**

- La jointure garde les 13 305 915 transactions, et aucune n'est sans catégorie.
- Les 109 codes ont une famille et une description en français.
- Les deux plus grandes familles sont « Transport, carburant et automobile » (3 352 312 transactions, 25,2 %) et « Alimentation » (3 182 334, 23,9 %). La plus petite est « Voyage et hébergement » (52 026, 0,4 %).

## P3-06 : transformation de la colonne errors

Critère de fin : totaux identiques à ceux de P2-13. Voir `docs/P3-06_colonne_errors.md`.

In [6]:
df_transaction = transform_errors_column(df_transaction)
check_errors_transformation(df_transaction)

=== Contrôle P3-06 : Transformation de la colonne 'errors' ===
  - Bad PIN                (err_bad_pin              ) :    32482 | Réf P2-13 :    32482 [OK]
  - Bad CVV                (err_bad_cvv              ) :     6241 | Réf P2-13 :     6241 [OK]
  - Bad Card Number        (err_bad_card_number      ) :     7925 | Réf P2-13 :     7925 [OK]
  - Bad Expiration         (err_bad_expiration       ) :     6295 | Réf P2-13 :     6295 [OK]
  - Bad Zipcode            (err_bad_zipcode          ) :     1138 | Réf P2-13 :     1138 [OK]
  - Insufficient Balance   (err_insufficient_balance ) :   131621 | Réf P2-13 :   131621 [OK]
  - Technical Glitch       (err_technical_glitch     ) :    26633 | Réf P2-13 :    26633 [OK]
  - Types d'erreur absents de ALL_ERROR_TYPES : []
  - Transactions avec >= 1 erreur (has_error)      :   211393
  - Transactions avec échec auth (has_auth_error)  :    53977
Critère de fin P3-06 validé : True



**Constats**

- Les totaux des 7 colonnes d'erreur sont identiques à ceux de P2-13.
- 211 393 transactions ont au moins une erreur, dont 53 977 avec un échec d'authentification.

## P3-07 : structuration du lieu des transactions

Critère de fin : plus aucun manquant inexpliqué dans les colonnes de lieu. Voir `docs/P3-07_lieu_transactions.md`.

In [7]:
df_transaction = structure_transaction_location(df_transaction)
check_transaction_location(df_transaction)

=== Contrôle P3-07 : Structuration du lieu des transactions ===
  - merchant_city          : NaN =      0 | 'UNKNOWN' (inexpliqués) =      0
  - merchant_state_clean   : NaN =      0 | 'UNKNOWN' (inexpliqués) =      0
  - merchant_country       : NaN =      0 | 'UNKNOWN' (inexpliqués) =      0
  - is_online              : NaN =      0 | 'UNKNOWN' (inexpliqués) =      0
  - is_foreign             : NaN =      0 | 'UNKNOWN' (inexpliqués) =      0
  - zip_clean              : NaN =      0 | 'UNKNOWN' (inexpliqués) =      0

Répartition par type de localisation :
  - En ligne (is_online=1)      :    1563700
  - À l'étranger (is_foreign=1) :      89006
  - Aux USA (physique)          :   11653209

  - Désaccord ville/canal (online_channel_mismatch=1) :       5788


**Constats**

- Les colonnes de lieu n'ont plus aucune valeur manquante ni « UNKNOWN ».
- 1 563 700 transactions sont en ligne, 89 006 à l'étranger et 11 653 209 sur place aux États-Unis.
- 5 788 transactions sont signalées par `online_channel_mismatch`.

## P3-08 : règles pour les valeurs suspectes

Les règles viennent du rapport de diagnostic (`docs/P2-16_rapport_diagnostic.md`). Aucune ligne n'est supprimée et aucune valeur n'est modifiée : chaque cas est signalé par un indicateur 0/1.

Critère de fin : chaque règle appliquée et tracée. Voir `docs/P3-08_valeurs_suspectes.md`.

In [8]:
df_transaction = flag_suspicious_transactions(df_transaction)
df_users = flag_suspicious_users(df_users)
df_card = flag_suspicious_cards(df_card)

check_suspicious_flags(df_transaction, df_users, df_card)

,table,indicateur,règle,lignes,lignes signalées
0,transactions,is_negative_amount,montant négatif,13305915,660049
1,transactions,is_zero_amount,montant nul,13305915,10639
2,transactions,is_repeated,seconde ligne d'un groupe répété sans erreur,13305915,15
3,users,suspicious_income,revenu annuel < 1 000 ou revenu par habitant nul,2000,15
4,cards,zero_credit_limit,limite de crédit nulle,6146,31


**Constats**

- 660 049 montants négatifs et 10 639 montants nuls sont signalés.
- 15 lignes sont signalées par `is_repeated` : une par groupe répété sans erreur, comme prévu dans P2-14.
- 15 clients sont signalés par `suspicious_income`. C'est le même nombre que les revenus par habitant nuls : les 8 revenus annuels inférieurs à 1 000 en font tous partie.
- 31 cartes sont signalées par `zero_credit_limit`.
- Aucune ligne n'est supprimée et aucune valeur n'est modifiée.

## P3-09 : colonnes sensibles et inutiles

Colonnes retirées : `card_number`, `cvv` et `address` (sensibles), `card_on_dark_web` (une seule valeur, voir P2-11).

Critère de fin : colonnes absentes des tables nettoyées. Voir `docs/P3-09_colonnes_retirees.md`.

In [9]:
df_card = drop_columns(df_card, COLUMNS_TO_DROP["cards"])
df_users = drop_columns(df_users, COLUMNS_TO_DROP["users"])

check_columns_removed({"cards": df_card, "users": df_users})

,table,colonne,encore présente
0,cards,card_number,False
1,cards,cvv,False
2,cards,card_on_dark_web,False
3,users,address,False


**Constats**

- `card_number`, `cvv` et `card_on_dark_web` ne sont plus dans la table des cartes.
- `address` n'est plus dans la table des clients.

## P3-10 : enregistrement des tables nettoyées

Les cinq tables sont enregistrées au format Parquet dans `data/clean/`, qui n'est pas versionné. Pour les relire : `load_clean_table("transactions")`.

Critère de fin : tables rechargées en quelques secondes. Voir `docs/P3-10_tables_nettoyees.md`.

In [10]:
tables = {"transactions": df_transaction, "cards": df_card, "users": df_users,
          "mcc": df_mcc, "fraud_labels": df_labels}
save_clean_tables(tables)

,lignes,colonnes,taille du fichier (Mo)
table,,,
transactions,13305915,30,320.0
cards,6146,11,0.1
users,2000,14,0.1
mcc,109,4,0.0
fraud_labels,8914963,2,41.0


In [11]:
# Les tables en mémoire sont supprimées, puis relues depuis data/clean/
del df_transaction, df_card, df_users, df_mcc, df_labels, tables

tables = {}
durees = {}
for name in ["transactions", "cards", "users", "mcc", "fraud_labels"]:
    debut = time.perf_counter()
    tables[name] = load_clean_table(name)
    durees[name] = round(time.perf_counter() - debut, 1)
pd.Series(durees, name="secondes de lecture")

transactions    5.8
cards           0.0
users           0.0
mcc             0.0
fraud_labels    0.2
Name: secondes de lecture, dtype: float64

**Constats**

- Les cinq tables sont enregistrées dans `data/clean/`. Le fichier des transactions fait 320,0 Mo et celui des étiquettes 41,0 Mo. Les trois autres font 0,1 Mo ou moins.
- La relecture prend 5,8 s pour les transactions, 0,2 s pour les étiquettes et 0,0 s pour les trois autres tables.

## P3-11 : contrôles rejoués sur les tables nettoyées

Les contrôles du diagnostic sont relancés sur les tables relues depuis `data/clean/`.

Critère de fin : tous les contrôles passent. Voir `docs/P3-11_controles_rejoues.md`.

In [12]:
lignes_avant = {name: n for name, (n, cols) in etat_avant.items()}
controles = check_clean_tables(tables, lignes_avant, ref_erreurs)
print("Tous les contrôles passent :", controles["statut"].all())
controles

Tous les contrôles passent : True


,contrôle,obtenu,statut
0,lignes de transactions,13305915,True
1,lignes de cards,6146,True
2,lignes de users,2000,True
3,lignes de mcc,109,True
4,lignes de fraud_labels,8914963,True
5,montants non numériques,aucun,True
6,montants manquants,0,True
7,dates non converties,aucune,True
8,dates manquantes,0,True
9,lignes orphelines (P2-02),0,True


**Constats**

- Les 18 contrôles passent.
- Aucune table n'a perdu de ligne. Les montants sont numériques et les dates converties, sans valeur manquante.
- 0 ligne orpheline, 0 client différent du titulaire de la carte, 0 code MCC sans famille, 0 lieu manquant ou « UNKNOWN ».
- Les totaux des erreurs sont égaux à ceux de P2-13. Les colonnes retirées sont absentes, et les cinq indicateurs de P3-08 sont présents.
- Aucun identifiant de transaction n'est en double, et chaque étiquette correspond à une transaction.

## P3-12 : résumé pour le journal des transformations

Le journal complet est dans `docs/P3-12_journal_transformations.md`.

In [13]:
transformation_summary(etat_avant, tables)

,lignes avant,lignes après,colonnes avant,colonnes après,colonnes ajoutées,colonnes retirées
table,,,,,,
transactions,13305915,13305915,12,30,"has_error, err_bad_pin, err_bad_cvv, err_bad_card_number, err_bad_expiration, err_bad_zipcode, err_insufficient_balance, err_technical_glitch, has_auth_error, is_online, online_channel_mismatch, merchant_country, merchant_state_clean, is_foreign, zip_clean, is_negative_amount, is_zero_amount, is_repeated",aucune
cards,6146,6146,13,11,zero_credit_limit,"card_number, cvv, card_on_dark_web"
users,2000,2000,14,14,suspicious_income,address
mcc,109,109,2,4,"famille, description_fr",aucune
fraud_labels,8914963,8914963,2,2,aucune,aucune


**Constats**

- Aucune table ne perd de ligne.
- `transactions` passe de 12 à 30 colonnes : 18 colonnes ajoutées, aucune retirée.
- `cards` passe de 13 à 11 colonnes : 1 ajoutée, 3 retirées.
- `users` garde 14 colonnes : 1 ajoutée, 1 retirée.
- `mcc` passe de 2 à 4 colonnes. `fraud_labels` ne change pas.